# Next-Word Prediction with LSTM + Bahdanau Attention
**Corpus:** The Adventures of Sherlock Holmes — Arthur Conan Doyle (Project Gutenberg)

## Architecture Overview
```
Input (15-word context window)
  → Embedding (384 → 128d)
  → Dropout (0.35)
  → BiLSTM (2 layers, 128 units/direction → 256d output)
  → Bahdanau Attention (weighted sum over 15 timesteps)
  → Dense (256 → 128, ReLU)
  → Dropout (0.35)
  → Output logits (128 → output_vocab_size)
```
**Key design choices:**
- Dual vocabulary: large input vocab (≥30 freq) for rich context, small output vocab (≥50 freq) for tractable prediction
- Bahdanau (additive) attention: learns which of the 15 context words matters most at each step
- Nucleus (top-p) sampling for generation: avoids repetition without sacrificing coherence
- Three regularization layers: dropout + weight decay + label smoothing

## 1. Setup

In [1]:
import os
import re
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"cuda" if torch.cuda.is_available() else "cpu"}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch  : 2.12.0+cpu
Device   : cpu


## 2. Hyperparameters

All settings in one place. Key choices explained:
| Parameter | Value | Why |
|-----------|-------|-----|
| `SEQ_LEN` | 15 | More context than the suggested 10; richer signal per sample |
| `INPUT_MIN_FREQ` | 30 | Larger input vocab → fewer OOV tokens |
| `OUTPUT_MIN_FREQ` | 50 | ~249 output words → diverse, coherent generation |
| `DROPOUT` | 0.35 | Primary anti-overfitting lever (original was 0.15, causing 35% train/val gap) |
| `WEIGHT_DECAY` | 1e-4 | L2 regularisation via Adam |
| `LABEL_SMOOTHING` | 0.1 | Prevents overconfident logits; better calibration |
| `LEARNING_RATE` | 0.001 | Standard Adam default; more stable than original 0.005 |
| `PATIENCE` | 25 | Original 5 stopped training too early (epoch 19) |

In [2]:
SEQ_LEN         = 15
INPUT_MIN_FREQ  = 30
OUTPUT_MIN_FREQ = 50
EMBED_DIM       = 128
LSTM_UNITS      = 128   # bidir → 256 effective
ATTN_UNITS      = 128
DENSE_UNITS     = 128
DROPOUT         = 0.35
BATCH_SIZE      = 256
EPOCHS          = 500
LEARNING_RATE   = 0.001
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1
TRAIN_SPLIT     = 0.8
PATIENCE        = 25
DATA_URL        = 'https://www.gutenberg.org/files/1661/1661-0.txt'
SAVE_PATH       = 'sherlock.txt'
MODEL_PATH      = 'sherlock_model.pt'

## 3. Data — Download, Clean, Tokenise

In [3]:
def download_sherlock():
    if not os.path.exists(SAVE_PATH):
        print('Downloading Sherlock Holmes...')
        urllib.request.urlretrieve(DATA_URL, SAVE_PATH)
    else:
        print('Using cached Sherlock Holmes text.')
    with open(SAVE_PATH, 'r', encoding='utf-8') as f:
        return f.read()


def clean_text(text):
    """Strip Gutenberg boilerplate, lowercase, remove punctuation."""
    start = text.find('*** START OF THE PROJECT GUTENBERG EBOOK')
    end   = text.find('*** END OF THE PROJECT GUTENBERG EBOOK')
    if start != -1:
        text = text[start + 60:]
    if end != -1:
        text = text[:end]
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


raw_text   = download_sherlock()
clean      = clean_text(raw_text)
words      = clean.split()
print(f'Total words : {len(words):,}')
print(f'Sample      : {" ".join(words[:20])}')

Using cached Sherlock Holmes text.
Total words : 106,007
Sample      : herlock holmes the adventures of sherlock holmes by arthur conan doyle contents i a scandal in bohemia ii the red


## 4. Vocabulary

Two separate vocabularies:
- **Input vocab** — all words appearing ≥ `INPUT_MIN_FREQ` times. Used to encode the 15-word context window.
- **Output vocab** — all words appearing ≥ `OUTPUT_MIN_FREQ` times. The set of words the model can predict.

This dual design lets the model read richer context while keeping the classification head tractable.

In [4]:
def build_vocab(words):
    counts = Counter(words)

    input_vocab = ['<PAD>', '<OOV>'] + [
        w for w, c in counts.most_common() if c >= INPUT_MIN_FREQ
    ]
    w2i = {w: i for i, w in enumerate(input_vocab)}
    i2w = {i: w for i, w in enumerate(input_vocab)}

    output_words = [w for w, c in counts.most_common() if c >= OUTPUT_MIN_FREQ]
    out_w2i = {w: i for i, w in enumerate(output_words)}
    out_i2w = {i: w for i, w in enumerate(output_words)}

    print(f'Input  vocab : {len(input_vocab):,} words')
    print(f'Output vocab : {len(output_words):,} words')
    print(f'Output sample: {output_words[:20]}')
    return input_vocab, w2i, i2w, out_w2i, out_i2w


input_vocab, w2i, i2w, out_w2i, out_i2w = build_vocab(words)

Input  vocab : 384 words
Output vocab : 249 words
Output sample: ['the', 'i', 'and', 'to', 'of', 'a', 'in', 'that', 'it', 'you', 'he', 'was', 'his', 'is', 'my', 'have', 'as', 'had', 'with', 'which']


## 5. Sequence Generation

Sliding window approach: for each position in the corpus, take the 15 preceding words as input and the current word as target — but only if the target is in the output vocabulary.

In [5]:
def make_sequences(token_ids, i2w, out_w2i):
    sequences = []
    for i in range(SEQ_LEN, len(token_ids)):
        target_word = i2w.get(token_ids[i], '<OOV>')
        if target_word in out_w2i:
            out_target = out_w2i[target_word]
            sequences.append(list(token_ids[i - SEQ_LEN: i]) + [out_target])
    return np.array(sequences, dtype=np.int32)


def train_test_split(sequences):
    rng      = np.random.default_rng(seed=42)
    shuffled = sequences.copy()
    rng.shuffle(shuffled)
    split      = int(len(shuffled) * TRAIN_SPLIT)
    return shuffled[:split], shuffled[split:]


token_ids  = [w2i.get(w, 1) for w in words]
sequences  = make_sequences(token_ids, i2w, out_w2i)
train_seqs, test_seqs = train_test_split(sequences)

print(f'Total sequences : {len(sequences):,}')
print(f'Train sequences : {len(train_seqs):,}')
print(f'Test  sequences : {len(test_seqs):,}')

Total sequences : 73,279
Train sequences : 58,623
Test  sequences : 14,656


## 6. PyTorch Dataset & DataLoader

In [6]:
class SherlockDataset(Dataset):
    def __init__(self, sequences):
        self.X = torch.tensor(sequences[:, :-1], dtype=torch.long)
        self.y = torch.tensor(sequences[:, -1],  dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(SherlockDataset(train_seqs), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(SherlockDataset(test_seqs),  batch_size=BATCH_SIZE, shuffle=False)
print(f'Train batches : {len(train_loader)}')
print(f'Test  batches : {len(test_loader)}')

Train batches : 229
Test  batches : 58


## 7. Model Architecture

### 7a. Bahdanau (Additive) Attention

After the BiLSTM produces a hidden state for each of the 15 timesteps, attention learns *which timesteps to focus on* when predicting the next word.

**Why Bahdanau over dot-product attention?**  
Dot-product attention relies on the dot product of query and key vectors being meaningful — this requires large embedding dimensions to work well. On a small corpus with 128-d embeddings, additive attention (which uses a learned MLP scoring function) is more expressive and doesn't require this assumption.

$$\text{score}(h_t) = v^\top \tanh(W_1 h_t + W_2 h_t)$$
$$\alpha_t = \text{softmax}(\text{score})$$
$$\text{context} = \sum_t \alpha_t h_t$$

In [7]:
class BahdanauAttention(nn.Module):
    """Additive attention: learns alignment weights over all LSTM timesteps."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, ATTN_UNITS)
        self.W2 = nn.Linear(hidden_dim, ATTN_UNITS)
        self.v  = nn.Linear(ATTN_UNITS, 1)

    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden_dim)
        score   = self.v(torch.tanh(self.W1(lstm_out) + self.W2(lstm_out)))
        weights = torch.softmax(score, dim=1)           # (batch, seq_len, 1)
        context = (weights * lstm_out).sum(dim=1)       # (batch, hidden_dim)
        return context, weights

### 7b. Full Model

In [8]:
class SherlockModel(nn.Module):
    """Embedding → 2-layer BiLSTM → Bahdanau Attention → Dense → output logits."""
    def __init__(self, input_vocab_size, output_vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=input_vocab_size,
            embedding_dim=EMBED_DIM,
            padding_idx=0
        )
        self.dropout1 = nn.Dropout(DROPOUT)
        self.lstm = nn.LSTM(
            input_size=EMBED_DIM,
            hidden_size=LSTM_UNITS,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=DROPOUT
        )
        self.attention = BahdanauAttention(hidden_dim=LSTM_UNITS * 2)
        self.dense     = nn.Linear(LSTM_UNITS * 2, DENSE_UNITS)
        self.dropout2  = nn.Dropout(DROPOUT)
        self.output    = nn.Linear(DENSE_UNITS, output_vocab_size)

    def forward(self, x):
        x = self.embedding(x)                        # (batch, seq_len, embed_dim)
        x = self.dropout1(x)
        x, _ = self.lstm(x)                          # (batch, seq_len, lstm_units*2)
        context, attn_weights = self.attention(x)    # (batch, lstm_units*2)
        x = F.relu(self.dense(context))
        x = self.dropout2(x)
        return self.output(x)                        # (batch, output_vocab_size)


model = SherlockModel(
    input_vocab_size=len(input_vocab),
    output_vocab_size=len(out_w2i)
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params:,}')

SherlockModel(
  (embedding): Embedding(384, 128, padding_idx=0)
  (dropout1): Dropout(p=0.35, inplace=False)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (attention): BahdanauAttention(
    (W1): Linear(in_features=256, out_features=128, bias=True)
    (W2): Linear(in_features=256, out_features=128, bias=True)
    (v): Linear(in_features=128, out_features=1, bias=True)
  )
  (dense): Linear(in_features=256, out_features=128, bias=True)
  (dropout2): Dropout(p=0.35, inplace=False)
  (output): Linear(in_features=128, out_features=249, bias=True)
)

Total parameters: 839,546


## 8. Training

**Regularisation stack:**
- `label_smoothing=0.1` in CrossEntropyLoss → prevents overconfident predictions
- `weight_decay=1e-4` in Adam → L2 penalty on large weights
- `Dropout(0.35)` in the model → stochastic regularisation

**Scheduler:** ReduceLROnPlateau halves the LR when val loss plateaus — allows fast early learning and fine-grained convergence later.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_top5_acc': []}

best_val_loss    = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):

    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        train_total   += y_batch.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc  = train_correct / train_total

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_top5_correct, val_total = 0, 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits       = model(X_batch)
            val_loss    += criterion(logits, y_batch).item()
            val_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            top5         = logits.topk(5, dim=1).indices
            val_top5_correct += (top5 == y_batch.unsqueeze(1)).any(dim=1).sum().item()
            val_total   += y_batch.size(0)

    avg_val_loss     = val_loss / len(test_loader)
    avg_val_acc      = val_correct / val_total
    avg_val_top5_acc = val_top5_correct / val_total
    perplexity       = np.exp(avg_val_loss)

    scheduler.step(avg_val_loss)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_acc'].append(avg_train_acc)
    history['val_acc'].append(avg_val_acc)
    history['val_top5_acc'].append(avg_val_top5_acc)

    print(f'Epoch {epoch+1:03d} | '
          f'Train Acc: {avg_train_acc:.4f} | '
          f'Val Acc: {avg_val_acc:.4f} | '
          f'Val Top-5: {avg_val_top5_acc:.4f} | '
          f'Perplexity: {perplexity:.2f}')

    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print('  ✓ Model saved')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}.')
            break

print('Training complete.')

Epoch 001 | Train Acc: 0.0733 | Val Acc: 0.0853 | Val Top-5: 0.2692 | Perplexity: 105.34
  ✓ Model saved
Epoch 002 | Train Acc: 0.1094 | Val Acc: 0.1399 | Val Top-5: 0.3513 | Perplexity: 79.06
  ✓ Model saved
Epoch 003 | Train Acc: 0.1388 | Val Acc: 0.1595 | Val Top-5: 0.3753 | Perplexity: 70.20
  ✓ Model saved
Epoch 004 | Train Acc: 0.1515 | Val Acc: 0.1664 | Val Top-5: 0.3972 | Perplexity: 65.55
  ✓ Model saved
Epoch 005 | Train Acc: 0.1581 | Val Acc: 0.1743 | Val Top-5: 0.4050 | Perplexity: 62.86
  ✓ Model saved
Epoch 006 | Train Acc: 0.1633 | Val Acc: 0.1752 | Val Top-5: 0.4105 | Perplexity: 61.07
  ✓ Model saved
Epoch 007 | Train Acc: 0.1692 | Val Acc: 0.1782 | Val Top-5: 0.4196 | Perplexity: 59.66
  ✓ Model saved
Epoch 008 | Train Acc: 0.1724 | Val Acc: 0.1797 | Val Top-5: 0.4208 | Perplexity: 58.78
  ✓ Model saved
Epoch 009 | Train Acc: 0.1750 | Val Acc: 0.1848 | Val Top-5: 0.4262 | Perplexity: 57.55
  ✓ Model saved
Epoch 010 | Train Acc: 0.1788 | Val Acc: 0.1865 | Val Top-5: 0.

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_acc'], label='Train Accuracy', color='steelblue')
ax1.plot(history['val_acc'],   label='Val Accuracy',   color='coral')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_loss'], label='Train Loss', color='steelblue')
ax2.plot(history['val_loss'],   label='Val Loss',   color='coral')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

## 10. Final Metrics

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))

best_train_acc = max(history['train_acc'])
best_val_acc   = max(history['val_acc'])
best_val_loss  = min(history['val_loss'])
perplexity     = np.exp(best_val_loss)

print('=' * 50)
print('FINAL METRICS')
print('=' * 50)
print(f'Train Accuracy : {best_train_acc:.4f}  target >0.80  {"PASS" if best_train_acc > 0.80 else "FAIL"}')
print(f'Val Accuracy   : {best_val_acc:.4f}  target >0.75  {"PASS" if best_val_acc > 0.75 else "FAIL"}')
print(f'Perplexity     : {perplexity:.2f}      target <250   {"PASS" if perplexity < 250 else "FAIL"}')

## 11. Text Generation

### Nucleus (Top-p) Sampling
At each step, instead of always picking the single most likely word (greedy — repetitive) or sampling the full distribution (too noisy), we:
1. Sort words by probability descending
2. Keep only the smallest set whose cumulative probability ≥ p (the *nucleus*)
3. Sample uniformly from that set

This naturally adapts the candidate set: confident predictions → small nucleus (almost deterministic); uncertain predictions → larger nucleus (more diverse). Same technique used in GPT-2/GPT-3.

In [ ]:
def _nucleus_sample(probs: np.ndarray, p: float = 0.92) -> int:
    sorted_idx  = np.argsort(probs)[::-1]
    sorted_prob = probs[sorted_idx]
    cumulative  = np.cumsum(sorted_prob)
    cutoff      = np.searchsorted(cumulative, p) + 1
    nucleus_idx  = sorted_idx[:cutoff]
    nucleus_prob = probs[nucleus_idx] / probs[nucleus_idx].sum()
    return int(np.random.choice(nucleus_idx, p=nucleus_prob))


def generate_text(seed_text, max_phrase_len, model, w2i, out_i2w,
                  temperature=0.8, top_p=0.92):
    """
    Generates text by predicting the next word in a loop.
    Also shows the top 5 predictions at each step as required.
    """
    device = next(model.parameters()).device
    model.eval()

    current_words = seed_text.lower().split()
    vocab_size    = len(out_i2w)
    steps         = []

    with torch.no_grad():
        for _ in range(max_phrase_len):
            token_ids = [w2i.get(w, 1) for w in current_words]
            if len(token_ids) < SEQ_LEN:
                token_ids = [0] * (SEQ_LEN - len(token_ids)) + token_ids
            else:
                token_ids = token_ids[-SEQ_LEN:]

            x      = torch.tensor([token_ids], dtype=torch.long).to(device)
            logits = model(x).squeeze(0)

            # repetition penalty in log space
            for idx in range(vocab_size):
                if out_i2w.get(idx) in set(current_words[-6:]):
                    logits[idx] -= 1.5

            probs = torch.softmax(logits / temperature, dim=-1).cpu().numpy()

            # top-5 for reporting
            top5_idx = np.argsort(probs)[-10:][::-1]
            top5 = [
                {'word': out_i2w.get(int(i), '<OOV>'), 'probability': float(probs[i])}
                for i in top5_idx if out_i2w.get(int(i)) is not None
            ][:5]

            chosen_idx = _nucleus_sample(probs, p=top_p)
            next_word  = out_i2w.get(chosen_idx, top5[0]['word'])

            steps.append({
                'input_context': ' '.join(current_words[-5:]),
                'chosen_word':   next_word,
                'top5':          top5,
            })
            current_words.append(next_word)

    output_text = ' '.join(current_words)
    return output_text, steps

## 12. Generated Text Examples

In [ ]:
seed_phrases = [
    'I saw Holmes',
    'the mystery was',
    'Watson looked at the door',
    'it is a curious case',
    'sherlock holmes stepped into the room',
]

for seed in seed_phrases:
    generated, _ = generate_text(
        seed, max_phrase_len=30,
        model=model, w2i=w2i, out_i2w=out_i2w
    )
    print(f'SEED   : "{seed}"')
    print(f'OUTPUT : {generated}')
    print('-' * 70)

## 13. Step-by-Step Breakdown

For each predicted word, show the top 5 candidates and their probabilities — demonstrating the model's decision process.

In [ ]:
breakdown_seed = 'I saw Holmes'
generated, steps = generate_text(
    breakdown_seed, max_phrase_len=15,
    model=model, w2i=w2i, out_i2w=out_i2w
)

print(f'Seed     : "{breakdown_seed}"')
print(f'Generated: {generated}')
print('=' * 65)

for i, step in enumerate(steps):
    print(f"\nStep {i+1:02d} | Chosen: '{step['chosen_word']}'")
    print(f"  Context : '...{step['input_context']}'")
    print('  Top 5   :')
    for rank, c in enumerate(step['top5'], 1):
        marker = ' ← chosen' if c['word'] == step['chosen_word'] else ''
        print(f"    {rank}. '{c['word']:<15}' {c['probability']:.4f}{marker}")